<a href="https://colab.research.google.com/github/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised/blob/main/stimuli_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trial-level stimulus analysis of sentence-final expectancy and semantic integration**

This notebook prepares the stimulus-analysis stage for a revised trial-level reanalysis of the Toffolo et al. (2022) N400 dataset.
The original study tested ERP responses to sentence-final congruent and incongruent words, with effects reported for the Recognition Potential (RP), N400, and Late Positive Component (LPC/P600).
The revised analysis keeps the same sentence-final focus but prepares richer item-level regressors for later linear mixed-effects modelling of ERP amplitudes. The aim is to move from condition-level contrasts toward trial/item-level modelling of continuous linguistic variation across subjects and stimuli.

This notebook belongs to the GitHub repository for the revised Toffolo et al. language reanalysis: [https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised](https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised)

The stimuli-analysis pipeline extracts predictors describing how each final word relates to its preceding sentence context. These include human cloze probability, GPT-2 surprisal, LLM-derived expectancy, human–LLM expectancy disagreement, semantic similarity, lexical frequency, word length, phonology, syntactic complexity, and affective features. This pipeline exports a combined TSV regressor table for later ERP analysis, together with intermediate per-file predictor tables and correlation/VIF diagnostics.
The combined TSV contains the trial/item-level stimulus regressors used to test whether expectancy, semantic fit, lexical properties, phonological structure, syntactic complexity, and affective tone explain variation in RP, N400/PN400, and LPC/P600 amplitudes.

The later ERP analyses use linear mixed-effects models because ERP amplitudes are measured repeatedly across trials, subjects, and stimulus items. This modelling approach allows continuous linguistic predictors to be analysed at the trial level while accounting for variability across participants and stimuli.

The main methodological hypothesis is that trial/item-level linear mixed-effects modelling will provide a cleaner analysis of ERP variability than condition-level averaging. The ERP hypotheses are that low-cloze or high-surprisal sentence endings will increase N400/PN400 amplitude, lexical properties may modulate RP responses, and semantic or syntactic deviations may contribute to LPC/P600 effects. The LLM analysis extends the original cloze-probability approach by comparing human-rated expectancy with model-derived surprisal and identifying where each measure adds explanatory value in the ERP models.

The following cells clone the GitHub repository, install the dependencies listed in `requirements.txt`, unzip `stimuli.zip`, and run `process_stimuli.py`. Running these cells reproduces the combined stimulus-level TSV output and the complementary diagnostic files.

In [ ]:
%cd /content

# !rm -rf Semantically_Incongruent_or_Congruent_Eggplants_revised
!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised

In [ ]:
!pip install -r requirements.txt

In [ ]:
!unzip stimuli/stimuli.zip

**Language Predictor Extraction**

The next stage of the analysis computes the linguistic and psycholinguistic predictors that will later be used as trial-level regressors in the ERP models. The aim is to characterise each sentence-final word using a range of continuous measures describing lexical properties, phonological structure, affective content, contextual predictability, semantic fit, and syntactic complexity.
The following sections compute these predictors step by step and add them to a common dataframe that will ultimately form the stimulus-regressor table used in the later ERP analyses.

In [ ]:
from pathlib import Path
import pandas as pd
import spacy # natural language processing (NLP) library
from stimuli_analysis.process_stimuli_stepwise import *

file_path = Path("N400Stimset_stimuli_parameters.tsv")

df = load_table(file_path)
prepared = start_processing_table(df, file_path)

out, text_col, target_col, context_col, condition_col = prepared

out = add_sentence_target_context_columns(
    out,
    file_path,
    text_col,
    target_col,
    context_col
)

Let's examine the `out` DataFrame. This DataFrame takes the origianl tsv and organize it such as the different linguisic metrics can be computed.

In [ ]:
print('Displaying the first 5 rows of the updated \'out\' DataFrame:')
display(out.head())
print('\nDisplaying the info of the updated \'out\' DataFrame to see column types and non-null counts:')
out.info()

**Lexical Frequency of target words**

This section computes lexical frequency (how often a specific word is used in a language) measures for each target word. The pipeline uses the wordfreq library to estimate both raw frequency (total count of a word's appearances in the wordfreq library) and Zipf frequency (logarithmic scale that measures word popularity on a standard 0-to-8 scale)values for the target word. Frequency is included because common and rare words are processed differently during language comprehension, making lexical frequency an important control variable when examining contextual and semantic effects.

In [ ]:
out = add_lexical_frequency_features(out)

import matplotlib.pyplot as plt
import seaborn as sns

# the new columns are named 'target_zipf_frequency' and 'target_raw_frequency'
zipf_col = 'target_zipf_frequency'
raw_col = 'target_raw_frequency'

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot histogram for Zipf frequency
sns.histplot(out[zipf_col], bins=10, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Target Word Zipf Frequency')
axes[0].set_xlabel('Zipf Frequency')
axes[0].set_ylabel('Count')
axes[0].grid(True, linestyle='--', alpha=0.7) # Add grid to Zipf plot

# Plot histogram for Raw frequency
sns.histplot(out[raw_col], bins=50, kde=True, ax=axes[1]) # More bins for raw frequency as it can have a wider range
axes[1].set_title('Distribution of Target Word Raw Frequency')
axes[1].set_xlabel('Raw Frequency')
axes[1].set_ylabel('Count')
axes[1].grid(True, linestyle='--', alpha=0.7) # Add grid to Raw plot

plt.tight_layout()
plt.show()

**Phonological Features of target words**

This section computes phonological properties of each target word. Using the CMU Pronouncing Dictionary through the pronouncing package, the pipeline extracts the number of phonemes, the number of syllables, and the onset phoneme for each word. These measures provide information about the sound structure of the target word and are included to capture variation related to word form rather than meaning or contextual predictability.

In [ ]:
out = add_phonology_features(out)

# The new columns for phonological features
phonemes_col = 'target_n_phonemes'
syllables_col = 'target_n_syllables'
onset_phoneme_col = 'target_onset_phoneme' # Assuming this is the column name for onset phoneme

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(21, 5)) # Adjusted for three plots

# Plot histogram for number of phonemes
sns.histplot(out[phonemes_col], bins=max(out[phonemes_col].astype(int)) - min(out[phonemes_col].astype(int)) + 1, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Target Word Number of Phonemes')
axes[0].set_xlabel('Number of Phonemes')
axes[0].set_ylabel('Count')
axes[0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for number of syllables
sns.histplot(out[syllables_col], bins=max(out[syllables_col].astype(int)) - min(out[syllables_col].astype(int)) + 1, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Target Word Number of Syllables')
axes[1].set_xlabel('Number of Syllables')
axes[1].set_ylabel('Count')
axes[1].grid(True, linestyle='--', alpha=0.7)

# Plot count plot for onset phoneme (assuming it's categorical)
sns.countplot(y=out[onset_phoneme_col], order=out[onset_phoneme_col].value_counts().index, ax=axes[2])
axes[2].set_title('Distribution of Target Word Onset Phoneme')
axes[2].set_xlabel('Count')
axes[2].set_ylabel('Onset Phoneme')
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Emotion Features of target words**

This section computes affective properties of the target word. The pipeline queries NRC emotion and VAD lexicons to obtain valence (positve or negative emotions), arousal (hogh to low physical or mental alertness), dominance (degree of control, power, or influence), emotion-category labels, and an overall emotionality indicator. These predictors are included because emotional content can influence language processing and neural responses independently of contextual expectancy or semantic fit.

*Note*: The emotional indicator is a binary variables computed as True if any of the following conditions are met for a given word:
- the emotion_count (sum of specific emotion flags like anger, disgust, fear, joy, sadness, surprise, trust) is greater than 0.
- the word is associated with 'positive' or 'negative' emotion categories
- the absolute difference between its valence or arousal score and 0.5 (the neutral point) is greater than 0.15. This indicates a significant deviation from neutrality in terms of pleasantness or intensity.

In [ ]:
out = add_target_emotion_features(out)

# These column names are added by add_target_emotion_features
valence_col = 'target_valence'
arousal_col = 'target_arousal'
dominance_col = 'target_dominance'
emotionality_indicator_col = 'target_is_emotional'

# Create a figure with 2x2 subplots for all emotion features
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot histogram for Valence (top-left)
sns.histplot(out[valence_col], bins=10, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Target Word Valence')
axes[0, 0].set_xlabel('Valence Score')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Arousal (top-right)
sns.histplot(out[arousal_col], bins=10, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of Target Word Arousal')
axes[0, 1].set_xlabel('Arousal Score')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Dominance (bottom-left)
sns.histplot(out[dominance_col], bins=10, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Target Word Dominance')
axes[1, 0].set_xlabel('Dominance Score')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, linestyle='--', alpha=0.7)

# Plot count plot for Overall Emotionality Indicator (bottom-right)
sns.countplot(x=out[emotionality_indicator_col], ax=axes[1, 1])
axes[1, 1].set_title('Distribution of Target Word Emotionality')
axes[1, 1].set_xlabel('Is Emotional (False/True)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Surprisal  of target words and Sentence Predictability**

This section computes model-based measures of contextual predictability using GPT-2. The pipeline calculates the surprisal of the target word given the preceding sentence context, together with sentence-level surprisal and sentence perplexity measures. Target-word surprisal is computed from the probability assigned by GPT-2 to the observed target word within its context, while sentence-level measures summarise the predictability of the sentence as a whole. Surpisal is the flip side of cloze probability (-log2(CP)).

These predictors are included because they provide quantitative estimates of contextual expectation that can later be compared with human cloze probability measures in the ERP analyses.

In [ ]:
from stimuli_analysis.surprisal import SurprisalModel
from stimuli_analysis.sentence_metrics import SentenceMetrics

surprisal_model = SurprisalModel(model_name="gpt2")

out = add_surprisal_features(
    out,
    file_path,
    text_col,
    surprisal_model
)

In [ ]:
target_surprisal_col = 'target_surprisal_bits'
sentence_surprisal_col = 'sentence_mean_surprisal'
sentence_perplexity_col = 'sentence_perplexity'

# Create a figure with three subplots for surprisal features
fig_surprisal, axes_surprisal = plt.subplots(1, 3, figsize=(20, 5))

# Plot histogram for Target Word Surprisal
sns.histplot(out[target_surprisal_col], bins=10, kde=True, ax=axes_surprisal[0])
axes_surprisal[0].set_title('Distribution of Target Word Surprisal')
axes_surprisal[0].set_xlabel('Surprisal (bits)')
axes_surprisal[0].set_ylabel('Count')
axes_surprisal[0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Sentence Surprisal
sns.histplot(out[sentence_surprisal_col], bins=10, kde=True, ax=axes_surprisal[1])
axes_surprisal[1].set_title('Distribution of Sentence Surprisal')
axes_surprisal[1].set_xlabel('Surprisal (bits)')
axes_surprisal[1].set_ylabel('Count')
axes_surprisal[1].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Sentence Perplexity
sns.histplot(out[sentence_perplexity_col], bins=10, kde=True, ax=axes_surprisal[2])
axes_surprisal[2].set_title('Distribution of Sentence Perplexity')
axes_surprisal[2].set_xlabel('Perplexity')
axes_surprisal[2].set_ylabel('Count')
axes_surprisal[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Cloze Probability Metrics**

This section computes cloze-related predictors using available human and language-model probability estimates. For human cloze probability, these measures are **loaded** from the input table (experimentally measures). For the LLM cloze probability, a pre-trained Large Language Model, specifically GPT-2, is used to estimate the probability of the target word given its preceding sentence context. Subsequently, the pipeline calculates human and model unexpectedness measures, derives disagreement metrics between human and model expectations, and creates standardised versions of these predictors. These measures are included because cloze probability is one of the most widely used indicators of contextual expectancy in language-comprehension research.

In [ ]:
out = add_final_cloze_features(out)

# 'cloze-probability%_div' is the existing human cloze
human_cloze_col = 'cloze-probability%_div'
# Corrected column names based on the output
model_cloze_col = 'llm_cp'
disagreement_col = 'abs_context_constraint_llm_disagreement'

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Convert human cloze probability to a 0-1 scale for plotting
out['human_cp_proportion'] = out[human_cloze_col] / 100.0

# Plot histogram for Human Cloze Probability
sns.histplot(out['human_cp_proportion'], bins=10, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Human Cloze Probability')
axes[0].set_xlabel('Cloze Probability (0-1)')
axes[0].set_ylabel('Count')
axes[0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Model Cloze Probability
sns.histplot(out[model_cloze_col], bins=10, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Model Cloze Probability')
axes[1].set_xlabel('Model Cloze Probability (0-1)') # Updated label
axes[1].set_ylabel('Count')
axes[1].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Human-Model Cloze Disagreement
sns.histplot(out[disagreement_col], bins=10, kde=True, ax=axes[2])
axes[2].set_title('Distribution of Human-Model Cloze Disagreement')
axes[2].set_xlabel('Cloze Disagreement')
axes[2].set_ylabel('Count')
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Human vs LLM**

Here, we perform a simple t-test comparing the normalized cloze probability obtained on humans vs. the GPT-2 values

**Context–Target Semantic Similarity of target words**

This section computes semantic similarity between the sentence context and the target word. While the cloze probability is the likelihood of the next word, the semantic similarity is how much it is in the same topic area. In natural context these are related but exparimentally this can be dissociated (i.e. prsent an unexpected word, low CP, but from the same semantic domain or not). A BERT-based embedding model is used to generate vector representations for the context and target, and cosine similarity is calculated between the resulting embeddings. This predictor provides a model-based estimate of semantic fit between the target word and its preceding context.

In [ ]:
semantic_model = SentenceMetrics(model_name="bert-base-uncased")

out = add_semantic_similarity_features(
    out,
    file_path,
    semantic_model
)

print("Columns after adding semantic similarity features:")
print(out.columns.tolist())

In [ ]:
semantic_similarity_col = 'context_target_similarity'

plt.figure(figsize=(8, 6))
sns.histplot(out[semantic_similarity_col], bins=10, kde=True)
plt.title('Distribution of Context-Target Semantic Similarity')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Sentence Syntactic Complexity**

This section computes sentence-level syntactic features using spaCy dependency parses. The pipeline extracts the token count (number of items), the dependency distance (the linear gap, ie number of intervening words, between a dependent word and its structural head word), parse depth (maximum number of structural links (hops) from the Root verb down to the most deeply nested word in the sentence tree), and the number of subordinate clauses. These predictors are included to capture variation in sentence structure that may contribute to processing difficulty independently of lexical or semantic factors.

In [ ]:
syntax_model = spacy.load("en_core_web_sm")

out = add_syntax_features(
    out,
    file_path,
    text_col,
    syntax_model
)

print("Columns after adding syntactic complexity features:")
print(out.columns.tolist())

In [ ]:
token_count_col = 'syntax_n_tokens'
dependency_distance_col = 'syntax_mean_dependency_distance'
parse_depth_col = 'syntax_max_parse_depth'
subordinate_clauses_col = 'syntax_n_subordinate_clauses'

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot histogram for Sentence Token Count
sns.histplot(out[token_count_col], bins=10, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Sentence Token Count')
axes[0, 0].set_xlabel('Number of Tokens')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Mean Dependency Distance
sns.histplot(out[dependency_distance_col], bins=10, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of Mean Dependency Distance')
axes[0, 1].set_xlabel('Mean Dependency Distance')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Max Parse Depth
sns.histplot(out[parse_depth_col], bins=10, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Max Parse Depth')
axes[1, 0].set_xlabel('Max Parse Depth')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Number of Subordinate Clauses
sns.histplot(out[subordinate_clauses_col], bins=5, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of Number of Subordinate Clauses')
axes[1, 1].set_xlabel('Number of Subordinate Clauses')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Derived Predictors**

This section creates additional predictors from previously computed variables. Z-scored versions of surprisal and semantic similarity are generated, and these standardised measures are combined to create a context-shift index. These derived predictors provide normalised measures that facilitate comparison across variables and summarise relationships between contextual predictability and semantic fit.

In [ ]:
out = add_derived_predictor_features(out)

# Corrected column name for z-scored surprisal
z_surprisal_col = 'z_target_surprisal'
z_similarity_col = 'z_context_target_similarity'
context_shift_col = 'context_shift_index'

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Plot histogram for Z-scored Target Surprisal
sns.histplot(out[z_surprisal_col], bins=10, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Z-scored Target Surprisal')
axes[0].set_xlabel('Z-score Surprisal')
axes[0].set_ylabel('Count')
axes[0].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Z-scored Context-Target Semantic Similarity
sns.histplot(out[z_similarity_col], bins=10, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Z-scored Context-Target Semantic Similarity')
axes[1].set_xlabel('Z-score Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].grid(True, linestyle='--', alpha=0.7)

# Plot histogram for Context Shift Index
sns.histplot(out[context_shift_col], bins=10, kde=True, ax=axes[2])
axes[2].set_title('Distribution of Context Shift Index')
axes[2].set_xlabel('Context Shift Index')
axes[2].set_ylabel('Count')
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Predictor Diagnostics**

This final section evaluates relationships among the computed predictors. Correlation matrices and variance inflation factors (VIFs) are calculated for the main predictor set. These diagnostics help identify redundancy and multicollinearity before the predictors are used in later statistical analyses.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the list of columns to include in the correlation matrix
selected_columns = [
    'target_dur(s)',
    'cloze-probability%_div',
    'n_words_sentence',
    'target_n_letters',
    'target_zipf_frequency',
    'target_n_phonemes',
    'target_valence',
    'target_arousal',
    'target_dominance',
    'llm_cp',
    'context_target_similarity',
    'target_surprisal_bits',
    'sentence_mean_surprisal',
    'sentence_perplexity',
    'syntax_n_tokens',
    'syntax_mean_dependency_distance',
    'syntax_max_parse_depth',
    'syntax_n_subordinate_clauses',
    'context_shift_index'
]

# Ensure only existing columns are selected and remove duplicates from the list
existing_selected_columns = list(dict.fromkeys([col for col in selected_columns if col in out.columns]))

# Create a DataFrame with only the selected columns
df_for_correlation = out[existing_selected_columns]

# Compute the correlation matrix
correlation_matrix_subset = df_for_correlation.corr()

# Display the correlation matrix using a heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(correlation_matrix_subset, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Selected Language Metrics')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Shape of the correlation matrix subset:", correlation_matrix_subset.shape)

### Variance Inflation Factors (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Reuse the selected_columns defined for the correlation matrix
# Make sure to only include numerical columns that exist in 'out'
# And exclude any columns that might cause issues (e.g., constant values if they exist)

# Create a list of columns for VIF calculation, excluding 'context_shift_index' due to multicollinearity
vif_selected_cols = [col for col in selected_columns if col != 'context_shift_index']

# Ensure only numerical columns from the new vif_selected_cols are used for VIF
numerical_selected_cols_for_vif = [col for col in vif_selected_cols if col in out.columns and pd.api.types.is_numeric_dtype(out[col])]

# Create a DataFrame with only the numerical selected columns for VIF
# Drop rows with NaN or inf values in these columns to avoid errors in VIF calculation
df_for_vif = out[numerical_selected_cols_for_vif].dropna()._get_numeric_data() # Ensure all columns are numeric after dropna

# Check if df_for_vif is not empty and has more than one column (VIF requires at least two predictors besides the constant)
if not df_for_vif.empty and df_for_vif.shape[1] > 1:
    # Add a constant to the DataFrame, as VIF calculation requires an intercept
    X = add_constant(df_for_vif)

    # Calculate VIF for each predictor
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns

    # Iterate through columns to calculate VIF, skipping 'const'
    vif_values = []
    for i in range(X.shape[1]):
        if X.columns[i] != 'const':
            vif_values.append(variance_inflation_factor(X.values, i))
        else:
            vif_values.append(None) # Assign None or 0 to 'const' VIF
    vif_data["VIF"] = vif_values

    # Remove the VIF for the constant term as it is not interpretable
    vif_data = vif_data.dropna(subset=['VIF'])

    # Sort the VIF values for better visualization
    vif_data = vif_data.sort_values(by='VIF', ascending=False).reset_index(drop=True)

    # Display VIF data
    print("Variance Inflation Factors (for selected_columns, excluding 'context_shift_index'):")
    display(vif_data)

    # Plot VIF values as a bar chart
    plt.figure(figsize=(12, 8))
    sns.barplot(x='VIF', y='feature', data=vif_data, palette='viridis')
    plt.title('Variance Inflation Factors for Selected Numerical Variables (excluding Context Shift Index)')
    plt.xlabel('VIF Value')
    plt.ylabel('Feature')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
elif df_for_vif.shape[1] <= 1:
    print("Not enough numerical columns from selected_columns (excluding 'context_shift_index') to compute VIF after dropping NaNs (need at least 2).")
else:
    print("No numerical columns found from selected_columns (excluding 'context_shift_index') or all rows contain NaN values after filtering.")

**Save Final Predictor Table**

This section saves the completed predictor table. The final output contains the original stimulus information together with all computed linguistic and psycholinguistic predictors.

In [ ]:
output_dir = Path("language_outputs")
output_dir.mkdir(exist_ok=True)

out.to_csv(
    output_dir / "ALL_language_metrics.tsv",
    sep="\t",
    index=False
)

In [ ]:
save_predictor_diagnostics(
    out,
    output_dir / "ALL_predictor_diagnostics"
)

In [ ]:
!zip -r language_outputs.zip language_outputs

In [ ]:
# uncomment to download zip file when ran though Google collab
# from google.colab import files
# files.download("language_outputs.zip")